# From light to predicted microrobot bending

**Executable end-to-end mixed-abstraction model with separately validated transistor blocks.**

Abhineet Agarwal · IIT Bombay · IEEE member: No · SSCS member: No  
SPDX-License-Identifier: Apache-2.0

Actual drift-diffusion PV tables → characterized schematic clock / first DFF → behavioral divider/reset and finite-resistance drivers → nonlinear electrochemistry with a finite Pt return → assumed bending mechanics.

**Full transistor-coupled controller startup failed and remains unresolved.** The fresh reduced-system runs below do not close that failure. PV/process parameters are conceptual, mechanics are assumptions, walking/contact forces are absent, and physical verification covers the clock macro, not a complete controller. Archived physical/transistor results are identified separately from fresh Python calculations.

This notebook embeds the actual layout picture, plots, tables and expandable complete numerical records. Machine-readable JSON, compressed reduced-system waveforms and supporting evidence remain bundled. No raw SPICE transient, local reference PDF or supplied measurement CSV is redistributed. See [scope](examples/microrobot/SUBMISSION_STATUS.md), [setup](SETUP.md), and [attribution](ATTRIBUTION.md).

In [ ]:
from pathlib import Path
from uuid import uuid4
from io import BytesIO
import hashlib, html, json, sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, display

ROOT = Path.cwd().resolve()
assert (ROOT / "opentcad/__init__.py").is_file(), "Start Jupyter in the complete exported project folder"
sys.path.insert(0, str(ROOT))
APP = ROOT / "examples/microrobot"
ARCHIVE = APP / "results"
from examples.microrobot.tcad_demo import run
OUTPUT = APP / "local_runs" / ("submission_" + uuid4().hex[:12])

def read(name):
    return json.loads((ARCHIVE / name).read_text())
def table(headers, rows):
    text = '<table><thead><tr>' + ''.join('<th>'+html.escape(str(x))+'</th>' for x in headers) + '</tr></thead><tbody>'
    text += ''.join('<tr>'+''.join('<td>'+html.escape(str(x))+'</td>' for x in row)+'</tr>' for row in rows)
    display(HTML(text+'</tbody></table>'))
def details(title, obj):
    display(HTML('<details><summary>'+html.escape(title)+'</summary><pre>'+html.escape(json.dumps(obj, indent=2))+'</pre></details>'))
def picture(path, width=1100):
    display(Image(filename=str(path), width=width))
def showfig(fig):
    buffer = BytesIO(); fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    display(Image(data=buffer.getvalue())); plt.close(fig)
def num(value, scale=1):
    return '—' if value is None else f'{value*scale:.6g}'
print("Python:", sys.version.split()[0])
print("Fresh output:", OUTPUT.relative_to(ROOT))
print("No DD, SPICE, or layout regeneration; actual archived DD and transistor evidence are reused.")

## The actual compact clock layout

This picture is rendered from **actual flat Magic paint**, excluding checkpaint. It shows the compact clock macro, its bias circuitry and routing. The paint bounding box is **464.615 × 84.465 µm**, about **39,243.706 µm²**. These are paint extents, not an accepted robot footprint. The comparison baseline spans 615.615 × 84.465 µm. Native and GDS-roundtrip DRC/LVS evidence and extracted RC follow later.

This is **not a full controller layout**. The remaining divider, supervisor, transistor drivers, PV isolation, actuator geometry and footprint closure are outside this macro.

In [ ]:
compact = read("clock_compact_placement/metrics.json")
picture(ARCHIVE / "clock_compact_placement/prototype_layout.png")
table(["Paint geometry", "Width µm", "Height µm", "Bounding area µm²"],
      [(name, num(v["width_um"]), num(v["height_um"]), num(v["bounding_box_area_um2"])) for name,v in compact["area"].items()])
details("Complete compact placement, geometry and extraction record", compact)


## Device input: actual DD curves and the calibration gap

The PV calculation is a conceptual 3 µm Si junction at 300 K, 660 nm Beer–Lambert illumination, abrupt assumed doping and ideal ohmic contacts. No foundry PV process, measured-device fit, sidewall calibration or broadband optical calibration is implied. The source arrays are actual OpenTCAD/DEVSIM drift-diffusion outputs, not the empirical terminal fit.

The 6 s system uses four logic cells and two series cells in each actuator half, all at an assumed 1080 µm² per cell: eight cells / 8640 µm² total. Equal current density under area scaling, electrical isolation and linear photocurrent superposition are assumptions. The original paper's supply topology and cell allocation are different.

Published approximate anchors are ~0.3 A/W and ~0.6 V per cell; the conceptual DD result remains near 0.1355 A/W and 0.461 V. Separate observed trace summaries and a fixed-structure reflection-only feasibility investigation are included below as **archived comparisons**, not calibration of this PV.

In [ ]:
pv = read("pv.json")
fig, axes = plt.subplots(1,2,figsize=(11,3.7),layout="constrained")
for label in ("dark","light"):
    v=np.asarray(pv[label]["voltage_V"]); i=np.asarray(pv[label]["delivered_A_per_um2"])
    axes[0].plot(v,i*1e12,label=label); axes[1].plot(v,v*i*1e12,label=label)
axes[0].set(xlabel="Cell voltage (V)",ylabel="Delivered current (pA/µm²)",ylim=(-100,180))
axes[1].set(xlabel="Cell voltage (V)",ylabel="Delivered power (pW/µm²)",ylim=(-20,35))
for ax in axes: ax.legend(); ax.grid(alpha=.25)
showfig(fig)
table(["Quantity","Conceptual DD","Published approximate anchor","Interpretation"], [
    ["One-sun responsivity A/W",num(pv["light"]["responsivity_A_W"]),"~0.3","Uncalibrated"],
    ["Cell Voc V",num(pv["light"]["voc_V"]),"~0.6","Uncalibrated"],
    ["Cell Isc pA/µm²",num(pv["light"]["isc_A_per_um2"],1e12),"Not substituted","Actual DD input"],
    ["Medium/fine Isc relative change",num(abs(pv["medium_mesh"]["isc_A_per_um2"]/pv["light"]["isc_A_per_um2"]-1)),"Not measurement agreement","Mesh comparison"]])
details("Complete archived DD dark/light/mesh/lifetime/half-light arrays and terminal audits",pv)
calibration=read("pv_calibration_feasibility/metrics.json")
picture(ARCHIVE/"pv_calibration_feasibility/dd_comparison.png",1000)
picture(ARCHIVE/"pv_calibration_feasibility/throughput_feasibility.png",1000)
table(["Reflection-only feasibility quantity","Archived value"],list(calibration["feasibility"].items()))
details("Complete calibration-feasibility and observed acquisition-order summaries",calibration)


## Fresh end-to-end mixed-abstraction execution

The next cell executes exactly three inexpensive Python model cases: 6 s nominal at 1 ms, 6 s refinement at 0.5 ms, and a 6 s all-dark negative control. Illumination is off before 0.05 s, on until 3 s, dark during 3–3.4 s, then restored. Clock frequency/current follows the bounded **schematic** TT lookup; extracted-clock evidence is separate. Remaining divider/reset behavior and finite 100 MΩ drivers are behavioral. Local double-layer/adsorption/faradaic branches and finite Pt return feed back to the finite PV supplies.

The nominal and fine states must be finite and reach 6 s. Startup, shutdown, recovery, quarter-cycle phase, bounded adsorption coverage, PV/clock domains, electrolyte KCL and energy residual checks are asserted. A 5% threshold applies to the reported principal timestep differences. This verifies the stated reduced model, not the unresolved transistor implementation.

In [ ]:
report=run(OUTPUT,dt_s=0.001,duration_s=6.0)
assert report["all_gates_pass"],report["gates"]
table(["Acceptance gate","Pass"],report["gates"].items())
table(["Case","Rows","Start s","Actual stop s","Requested stop s","Complete","Finite"],
      [(k,v["rows"],v["start_s"],v["actual_stop_s"],v["requested_stop_s"],v["complete"],v["finite"]) for k,v in report["completion"].items()])
metrics=["mean_gait_Hz","startup_enable_s","steady_logic_V","steady_actuator_span_V","bending_peak_to_peak_rad","measured_phase_B_lag_cycles","electrical_energy_balance_relative_residual","electrolyte_KCL_peak_A"]
table(["Metric (units in name)","Nominal 1 ms","Fine 0.5 ms"],[(k,num(report["nominal"][k]),num(report["fine"][k])) for k in metrics])
table(["Principal refinement metric","Relative change","Threshold"],[(k,num(v),"0.05") for k,v in report["timestep_relative_changes"].items()])
table(["All-dark quantity","Maximum absolute value"],report["all_dark_max_abs"].items())
details("Complete fresh nominal/fine/dark summary, every metric and acceptance gate",report)


In [ ]:
picture(OUTPUT/"system.png")
wave=np.genfromtxt(OUTPUT/"waveforms.csv.gz",delimiter=",",names=True)
t=wave["time_s"]
fig,axes=plt.subplots(3,2,figsize=(12,10),layout="constrained")
for leg in ("A","B"):
    axes[0,0].plot(t,wave[f"theta_{leg}"],label=leg)
    axes[0,1].plot(t,wave[f"current_{leg}_A"]*1e9,label=leg)
axes[0,0].plot(t,wave["theta_return"],label="Pt return")
axes[0,1].plot(t,wave["current_return_A"]*1e9,label="Pt return")
axes[1,0].plot(t,wave["eta_return_V"],label="return interface")
axes[1,0].plot(t,wave["electrolyte_bulk_V"],label="solution bulk")
axes[1,1].plot(t,wave["electrolyte_KCL_A"]*1e18,label="KCL residual")
axes[2,0].plot(t,wave["storage_J"]*1e9,label="storage")
axes[2,1].plot(t,wave["phase_cycles"],label="phase accumulator")
for ax,ylabel in zip(axes.flat,["Adsorption coverage θ","Branch current nA","Potential V vs Pt","KCL residual aA","Stored energy nJ","Accumulated phase cycles"]):
    ax.set(xlabel="Time s",ylabel=ylabel); ax.axvspan(3,3.4,color="0.7",alpha=.2); ax.legend(); ax.grid(alpha=.2)
showfig(fig)
print("Nominal enable rises s:",report["nominal"]["enable_rises_s"],"; falls s:",report["nominal"]["enable_falls_s"])


## Energy accounting, design assumptions and provenance

Energy columns below retain their model meanings. “Bias” is the characterized analog-clock load in this coupling; logic includes the first-DFF lookup plus assumed remaining digital load. Adsorption is signed net electrical work, not heat. Adsorption/interface quantities include storage/reversible work and must **not** be added indiscriminately to dissipation. PV supply energies and final capacitor storage are separate. The residual is a numerical accounting check, not measured efficiency.

Mechanical relaxation is assumed to be 50 ms, curvature span 10⁵ m⁻¹ and hinge length 10 µm. The model predicts a bend angle, with no force, contact, friction, stride or displacement model.

In [ ]:
parts=["bias","logic","driver","electrolyte","faradaic","interface","adsorption","logic_PV","actuator_PV"]
table(["Model energy term","Mean power nW","Energy nJ/cycle"],[(k,num(report["nominal"][k+"_mean_W"],1e9),num(report["nominal"][k+"_J_per_cycle"],1e9)) for k in parts])
fig,ax=plt.subplots(figsize=(11,3.5),layout="constrained")
ax.bar(parts,[report["nominal"][k+"_J_per_cycle"]*1e9 for k in parts]);ax.set(ylabel="Model energy nJ/cycle",title="Separate accounting terms; storage/work terms are not additive losses")
ax.tick_params(axis="x",rotation=30);showfig(fig)
table(["Parameter (units in name)","Value"],report["parameters"].items())
details("Cell allocation, model boundaries, exact input hashes and execution settings",{k:report[k] for k in ["design","model_limits","provenance","execution"]})
details("Archived model development attempts: three calculations before report serialization fix + three accepted",read("tcad_demo/execution_history.json"))
print("This notebook independently reruns three model cases; no DD/SPICE/layout run is performed.")


## Separate transistor clock and extracted-layout comparisons

All cases below reuse archived full-transient audits. Their measured powers use full adaptive data over exactly 2.4–4 ms, with interpolated boundaries; thresholds follow instantaneous vdd/2. They use finite **fitted-PV** supplies at assumed cell scaling, unlike the conceptual DD-powered system above. Nominal comparisons keep source/model/load/startup settings fixed; routing and placement interventions are labeled. The 85 °C case changes electronics temperature while explicitly holding fitted PV at 26.85 °C.

The original RC layout slowed substantially versus schematic. Shorter stage-local routes and compact placement improved this bench. A passing compact bench does not establish complete-controller feasibility or footprint acceptance. The earlier matched-review README retains its historical preliminary status. These cases do not provide full PVT/mismatch qualification.

In [ ]:
clock_fine=read("clock_compact_convergence/metrics.json")
clock_hot=read("clock_compact_hot85/metrics.json")
cases=[("Schematic",25,26.85,compact["cases"]["schematic_25ns"]),
       ("Original extracted RC",25,26.85,compact["cases"]["extracted_25ns"]),
       ("Shorter stage-local routes",25,26.85,compact["cases"]["stage_local_25ns"]),
       ("Compact placement",25,26.85,compact["cases"]["compact_x_25ns"]),
       ("Compact fine",12.5,26.85,clock_fine["cases"]["compact_12p5ns"]),
       ("Compact hot, PV held 26.85°C",12.5,85,clock_hot["cases"]["electronics85_fixedPV26p85_12p5ns"])]
table(["Case","Max step ns","Electronics °C","Clock Hz","Analog nW","First DFF nW","Whole nW","Mean rail V","Rail min V","Rows","Stop s","Bench pass"],
      [(name,step,temp,num(v["edges"]["steady_clock_frequency_Hz"]),num(v["power_W"]["analog_clock"],1e9),num(v["power_W"]["first_ff_and_feedback"],1e9),num(v["power_W"]["whole_clock_and_first_ff"],1e9),num(v["rail_V"]["time_average"]),num(v["rail_V"]["minimum"]),v["rows"],v["actual_stop_s"],v["existing_bench_criterion_evaluation"]["passed"]) for name,step,temp,v in cases])
table(["25 ns → 12.5 ns observable","Coarse","Fine","Absolute relative difference","Threshold","Pass"],[(k,num(v["coarse_25ns"]),num(v["fine_12p5ns"]),num(v["absolute_relative_difference_to_fine"]),v["threshold"],v["passed"]) for k,v in clock_fine["comparisons"].items()])
table(["Hot sensitivity observable","Nominal","Hot","Hot / nominal"],[(k,num(v["nominal"]),num(v["hot85_fixed_PV"]),num(v["hot_over_nominal"])) for k,v in clock_hot["comparisons"].items()])
for title,record in [("Original matched pre/post-layout full-data audit",read("clock_layout_review/review_metrics.json")),("Shorter routes complete record",read("clock_local_routes/metrics.json")),("Compact clock timestep full-data audit",clock_fine),("Hot sensitivity full-data audit and fixed-PV temperature policy",clock_hot),("Schematic voltage/corner characterization used by the system lookup",read("clock_final/summary.json"))]: details(title,record)


## Physical gates and standalone transistor drivers

Physical gates below concern the compact clock macro only. DRC/LVS and negative controls assess the specific extracted connectivity; the paint bounding box is not a footprint signoff. No complete-controller layout is claimed.

Standalone MOS-driver tests use finite actuator PV and reduced electrode interfaces, but **ideal logic, phase and ready sources**. Their passing preliminary excursion/error limits do not establish transistor-driver behavior inside the nonlinear coupled startup. The fresh system uses finite-resistance behavioral drivers instead.

In [ ]:
physical=compact["physical_gates"]
table(["Clock-macro physical gate / inventory","Recorded value"],[(k,physical[k]) for k in ["placement_drc","native_drc_pass","native_lvs_pass","negative_control_rejected","roundtrip_drc_pass","roundtrip_lvs_pass","rc_resistors","rc_capacitors","inventory"]])
details("Complete native/roundtrip compact physical report",read("clock_compact_placement/physical/report.json"))
drivers=read("electrical_v2/summary.json")
rows=list(drivers["drivers"])+[drivers[k] for k in ["driver_timestep_half","driver_slow_inputs","driver_hot","driver_half_light"]]
table(["Corner","Temp °C","Light W/m²","Max step s","Settled error mV","Clamp error mV","PV nW","Logic nW","Pass"],[(v["corner"],v["temperature_C"],v["irradiance_W_m2"],v["timestep_s"],num(v["settled_output_error_V"],1e3),num(v["reset_clamp_error_after_20ms_V"],1e3),num(v["actuator_PV_mean_W"],1e9),num(v["driver_logic_supply_mean_W"],1e9),v["passed"]) for v in rows])
picture(ARCHIVE/"electrical_v2/drivers.png",1000)
details("Complete standalone-driver and rejected-brownout summary",drivers)


## Native integration failures and diagnostic boundaries

This chronology preserves failed and limited evidence. A complete small-signal, linear, held-reset, or isolated DC/AC diagnostic is not released nonlinear actuation. No sequence of these results closes the original native startup failure or uniquely identifies a simulator defect/root cause. The complete status/acceptance records are expandable below; raw local transient paths/hashes can appear as historical references, but raw transients are not bundled.

The final row is the fresh mixed-abstraction demonstration above. It succeeds under its explicit behavioral substitutions; the fully transistor-coupled controller remains unresolved.

In [ ]:
history=[("Initial TCAD controller","tcad_controller"),("Held-reset logic isolation","tcad_logic_isolation"),("Held-input actuator add-back","tcad_actuator_startup"),("Passive held-input network","tcad_passive_clamps"),("Charge adsorption reformulation","tcad_charge_adsorption"),("Passive small-signal adsorption","tcad_passive_adsorption"),("Stable nonlinear formulation","tcad_stable_adsorption"),("Linear behavioral-source control","tcad_linear_bads"),("Isolated native DC/AC source-law audit","tcad_adsorption_source_audit"),("Reduced actuator-feedback replay","tcad_actuator_feedback_replay"),("Saved feedback-control outcomes","tcad_feedback_fix")]
rows=[]
for label,folder in history:
    status=read(folder+"/status.json"); metric=read(folder+"/metrics.json")
    if folder=="tcad_controller": metric=metric["nominal_25ns"]
    rows.append((label,status.get("status"),metric.get("complete",metric.get("passed","separate outcomes")),metric.get("actual_stop_s","—"),metric.get("requested_stop_s","—")))
    details(label+": full status and numerical record",{"status":status,"metrics":read(folder+"/metrics.json")})
rows.append(("Fresh mixed-abstraction model","15 gates pass; behavioral substitutions explicit",report["all_gates_pass"],6.0,6.0))
table(["Investigation","Recorded status","Completion / local check","Actual stop s","Requested stop s"],rows)


## Full records, reproduction and remaining work

The notebook carries every fresh summary metric, exact model parameter, input hash, gate, nominal/fine/dark completion record, archived clock comparison, compact geometry/physical report, standalone driver summary and selected integration-failure record in its saved outputs. Expand any “Complete…” disclosure above to inspect JSON without rerunning. The full reduced-system time series is also available as compressed CSV.

The compact clock is separate from the schematic lookup used by the coupled system. PV calibration, cell isolation, capacitor area, practical protection/level shifting, full-controller layout, PVT/mismatch, force/contact mechanics and locomotion remain open. A half-timestep comparison at one condition is not universal convergence. The hot clock sensitivity does not resolve the prior hot coupled-controller failure.

Python 3.12 packages are pinned; third-party native dependencies remain external. Colab/Linux execution are untested. Historical figure-derived and supplied measurement CSVs are excluded from this export. Paper-derived summaries do not transfer rights in source data; see [attribution](ATTRIBUTION.md). No paper download, measurement fitting, DD/SPICE simulation or physical extraction is executed here.

`submission.json` identifies packaged content. The executor verifies source/reference hashes before and after execution and records the actual environment. Fresh outputs remain under `local_runs/`. The intended folder is `ISSCC27/submitted_notebooks/opentcad_microrobot/`; the October 31, 2026, 11:59 AM Pacific deadline has passed. This package makes no eligibility claim. Nothing has been submitted, pushed, published or sent for fabrication.

In [ ]:
from IPython.display import FileLink
for path in [OUTPUT/"summary.json",OUTPUT/"waveforms.csv.gz",ARCHIVE/"pv.json",ARCHIVE/"clock_compact_placement/metrics.json",ARCHIVE/"clock_compact_convergence/metrics.json",ARCHIVE/"clock_compact_hot85/metrics.json",ARCHIVE/"pv_calibration_feasibility/metrics.json",ARCHIVE/"electrical_v2/summary.json"]:
    display(FileLink(str(path.relative_to(ROOT))))
print("All 15 fresh model gates passed; all source/reference records above are archived evidence unless explicitly labeled fresh.")